# PASUL 1 — Lesion-level Split 80/10/10

**Scop:** Împarte imaginile în train_pool / val / test, cu garanții stricte împotriva data leakage:

1. **Lesion-level grouping** — toate imaginile aceleași leziuni rămân împreună (în train, val SAU test, niciodată split)
2. **Stratificare pe clasă** — distribuția celor 7 clase (mel, nv, bcc, akiec, bkl, df, vasc) e identică în toate splitturile
3. **Test set fix** — va fi folosit identic pentru toate cele 5 tier-uri din PASUL 2

**Justificare 80/10/10 (vs Nawaz et al. 90/10):**

> Spre deosebire de splitul 90/10 folosit de Nawaz et al. (care nu separă explicit validation de test, permițând hyperparameter tuning leakage), această lucrare utilizează un split lesion-level stratificat 80/10/10 cu seturi de validation și test distincte. Acest design asigură că performanța finală raportată reflectă generalizare reală, nu adaptare la setul de evaluare. În plus, gruparea lesion-level previne leakage prin imagini multiple ale aceleași leziuni — o sursă majoră de inflație artificială identificată în literatura de specialitate (Tschandl et al., 2018; Combalia et al., 2019).

**Output:**
- `train_pool.csv` — ~80% (input pentru PASUL 2 / construire tier-uri)
- `val_fixed.csv` — ~10% (validation set, fix pentru toate experimentele)
- `test_fixed.csv` — ~10% (test set, fix pentru toate experimentele)
- `step1_report.md` — raport cu distribuții și verificări de no-leakage

---

## 1. Setup

In [18]:
from google.colab import drive
drive.mount('/content/drive')
IN_COLAB = True

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:
BASE_DIR = '/content/drive/MyDrive/Thesis'
UNIFIED_METADATA_PATH = f'{BASE_DIR}/pipeline_outputs/step0/unified_metadata.csv'
OUTPUT_DIR = f'{BASE_DIR}/pipeline_outputs/step1'

TRAIN_RATIO = 0.80
VAL_RATIO   = 0.10
TEST_RATIO  = 0.10
RANDOM_SEED = 42

assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-9, "Ratios must sum to 1.0"

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Input: {UNIFIED_METADATA_PATH}")
print(f"Output: {OUTPUT_DIR}")
print(f"Split: {TRAIN_RATIO}/{VAL_RATIO}/{TEST_RATIO}")
print(f"Random seed: {RANDOM_SEED}")

Input: /content/drive/MyDrive/Thesis/pipeline_outputs/step0/unified_metadata.csv
Output: /content/drive/MyDrive/Thesis/pipeline_outputs/step1
Split: 0.8/0.1/0.1
Random seed: 42


## 2. Imports

In [20]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)

## 3. Citire unified_metadata

In [21]:
df = pd.read_csv(UNIFIED_METADATA_PATH)
print(f"Total imagini: {len(df):,}")
print(f"Lesion_id-uri unice: {df['lesion_id'].nunique():,}")
print(f"Clase unice: {df['dx'].nunique()}")
df.head(3)

Total imagini: 25,331
Lesion_id-uri unice: 13,931
Clase unice: 7


,image_id,dx_isic,dx,lesion_id,source,image_path,age,sex,anatom_site
0,ISIC_0000000,NV,nv,SYNTH_ISIC_0000000,other,/content/drive/MyDrive/Thesis/data/ISIC2019/IS...,55.0,female,anterior torso
1,ISIC_0000001,NV,nv,SYNTH_ISIC_0000001,other,/content/drive/MyDrive/Thesis/data/ISIC2019/IS...,30.0,female,anterior torso
2,ISIC_0000002,MEL,mel,SYNTH_ISIC_0000002,other,/content/drive/MyDrive/Thesis/data/ISIC2019/IS...,60.0,female,upper extremity


## 4. Agregare la nivel de leziune

Splitul se face la nivel de **leziune**, nu de imagine. Asta înseamnă că pentru fiecare `lesion_id` unic creăm un singur rând care conține:
- Clasa (`dx`) — toate imaginile unei leziuni au aceeași clasă, deci e safe să luăm prima
- Sursa (`source`) — la fel
- Numărul de imagini

Apoi facem split pe acest DataFrame de leziuni, iar la final expandăm înapoi la imagini.

In [22]:
class_per_lesion = df.groupby('lesion_id')['dx'].nunique()
inconsistent = class_per_lesion[class_per_lesion > 1]

if len(inconsistent) > 0:
    print(f"[ATENȚIE] {len(inconsistent)} leziuni au imagini cu clase diferite!")
    print("Asta nu ar trebui să se întâmple. Exemple:")
    for lid in inconsistent.head(5).index:
        classes = df[df['lesion_id'] == lid]['dx'].unique()
        print(f"  {lid}: {classes.tolist()}")
else:
    print("[OK] Fiecare leziune are exact o clasă consistentă peste imaginile sale.")

[OK] Fiecare leziune are exact o clasă consistentă peste imaginile sale.


In [23]:
lesion_df = df.groupby('lesion_id').agg(
    dx=('dx', 'first'),
    source=('source', 'first'),
    n_images=('image_id', 'count')
).reset_index()

print(f"Lesions totale: {len(lesion_df):,}")
print(f"\nDistribuție imagini per leziune:")
print(lesion_df['n_images'].value_counts().sort_index().to_string())
print(f"\nMedian: {lesion_df['n_images'].median():.0f}")
print(f"Max: {lesion_df['n_images'].max()}")
lesion_df.head(5)

Lesions totale: 13,931

Distribuție imagini per leziune:
n_images
1     8872
2     2505
3     1306
4      468
5      248
6      198
7      112
8       59
9       54
10      24
11      18
12      19
13      10
14       7
15       3
16       3
17       5
18       3
19       6
20       6
21       1
24       1
26       1
27       1
31       1

Median: 1
Max: 31


,lesion_id,dx,source,n_images
0,BCN_0000001,akiec,BCN_20000,3
1,BCN_0000002,nv,BCN_20000,3
2,BCN_0000003,akiec,BCN_20000,2
3,BCN_0000004,bcc,BCN_20000,6
4,BCN_0000008,bcc,BCN_20000,3


## 5. Stratified split 80/10/10 pe leziuni

Stratificare pe `dx` (clasa). Sklearn nu permite split direct 3-way, deci facem 2 splituri consecutive:

1. **Primul split:** 80% train_pool / 20% temp_pool
2. **Al doilea split:** 50/50 din temp_pool → val / test (rezultă 10/10 din total)

Ambele splituri folosesc același `random_state` pentru reproducibilitate.

In [24]:
# Verifică că fiecare clasă are suficiente leziuni pentru split stratificat
print("Distribuție clase pe leziuni:")
class_counts = lesion_df['dx'].value_counts()
print(class_counts.to_string())

# Cu split 80/10/10, clasa cea mai mică ar trebui să aibă cel puțin 10 leziuni
# pentru a avea măcar 1 reprezentant în val și test
min_class_size = class_counts.min()
print(f"\nClasa cea mai mică: {class_counts.idxmin()} cu {min_class_size} leziuni")
if min_class_size < 10:
    print("[ATENȚIE] Clasa cea mai mică are <10 leziuni — splitul stratificat poate să eșueze.")
else:
    print(f"[OK] Toate clasele au suficiente leziuni pentru stratify (min: {min_class_size}).")

Distribuție clase pe leziuni:
dx
nv       8644
mel      1676
bkl      1467
bcc      1310
akiec     586
vasc      135
df        113

Clasa cea mai mică: df cu 113 leziuni
[OK] Toate clasele au suficiente leziuni pentru stratify (min: 113).


In [25]:
# Split 1: 80% train / 20% temp
train_lesions, temp_lesions = train_test_split(
    lesion_df,
    test_size=(VAL_RATIO + TEST_RATIO),  # 20%
    stratify=lesion_df['dx'],
    random_state=RANDOM_SEED
)

# Split 2: 50/50 din temp -> val / test (10/10 din total)
val_lesions, test_lesions = train_test_split(
    temp_lesions,
    test_size=TEST_RATIO / (VAL_RATIO + TEST_RATIO),  # 0.5
    stratify=temp_lesions['dx'],
    random_state=RANDOM_SEED
)

print(f"Split rezultat (lesion-level):")
print(f"  Train: {len(train_lesions):,} leziuni  ({100*len(train_lesions)/len(lesion_df):.1f}%)")
print(f"  Val:   {len(val_lesions):,} leziuni  ({100*len(val_lesions)/len(lesion_df):.1f}%)")
print(f"  Test:  {len(test_lesions):,} leziuni  ({100*len(test_lesions)/len(lesion_df):.1f}%)")

Split rezultat (lesion-level):
  Train: 11,144 leziuni  (80.0%)
  Val:   1,393 leziuni  (10.0%)
  Test:  1,394 leziuni  (10.0%)


## 6. Expandare înapoi la nivel de imagine

Acum că avem leziunile împărțite, expandăm fiecare leziune la toate imaginile sale.
Rezultatul: 3 DataFrame-uri cu imaginile efective.

In [26]:
train_pool_df = df[df['lesion_id'].isin(train_lesions['lesion_id'])].copy()
val_df        = df[df['lesion_id'].isin(val_lesions['lesion_id'])].copy()
test_df       = df[df['lesion_id'].isin(test_lesions['lesion_id'])].copy()

# Adaugă coloana 'split' pentru audit
train_pool_df['split'] = 'train_pool'
val_df['split']        = 'val'
test_df['split']       = 'test'

print(f"Split rezultat (image-level):")
print(f"  Train pool: {len(train_pool_df):,} imagini  ({100*len(train_pool_df)/len(df):.2f}%)")
print(f"  Val:        {len(val_df):,} imagini  ({100*len(val_df)/len(df):.2f}%)")
print(f"  Test:       {len(test_df):,} imagini  ({100*len(test_df)/len(df):.2f}%)")
print(f"  TOTAL:      {len(train_pool_df) + len(val_df) + len(test_df):,} (vs {len(df):,} input)")

Split rezultat (image-level):
  Train pool: 20,299 imagini  (80.14%)
  Val:        2,519 imagini  (9.94%)
  Test:       2,513 imagini  (9.92%)
  TOTAL:      25,331 (vs 25,331 input)


## 7. Sanity checks: ZERO data leakage

In [27]:
# Check 1: niciun lesion_id nu apare în două splitturi
train_lids = set(train_pool_df['lesion_id'])
val_lids   = set(val_df['lesion_id'])
test_lids  = set(test_df['lesion_id'])

train_val_overlap  = train_lids & val_lids
train_test_overlap = train_lids & test_lids
val_test_overlap   = val_lids & test_lids

print("Verificare overlap între splitturi (lesion_id):")
print(f"  Train ∩ Val:  {len(train_val_overlap)} leziuni")
print(f"  Train ∩ Test: {len(train_test_overlap)} leziuni")
print(f"  Val ∩ Test:   {len(val_test_overlap)} leziuni")

assert len(train_val_overlap) == 0, "LEAKAGE: lesion_id apare în train și val!"
assert len(train_test_overlap) == 0, "LEAKAGE: lesion_id apare în train și test!"
assert len(val_test_overlap) == 0, "LEAKAGE: lesion_id apare în val și test!"

print("\n[OK] ZERO leakage la nivel de leziune.")

Verificare overlap între splitturi (lesion_id):
  Train ∩ Val:  0 leziuni
  Train ∩ Test: 0 leziuni
  Val ∩ Test:   0 leziuni

[OK] ZERO leakage la nivel de leziune.


In [28]:
# Check 2: niciun image_id nu apare în două splitturi (verificare suplimentară)
train_iids = set(train_pool_df['image_id'])
val_iids   = set(val_df['image_id'])
test_iids  = set(test_df['image_id'])

print("Verificare overlap între splitturi (image_id):")
print(f"  Train ∩ Val:  {len(train_iids & val_iids)} imagini")
print(f"  Train ∩ Test: {len(train_iids & test_iids)} imagini")
print(f"  Val ∩ Test:   {len(val_iids & test_iids)} imagini")

assert (train_iids & val_iids) == set(), "LEAKAGE: image_id în train și val!"
assert (train_iids & test_iids) == set(), "LEAKAGE: image_id în train și test!"
assert (val_iids & test_iids) == set(), "LEAKAGE: image_id în val și test!"

print("\n[OK] ZERO leakage la nivel de imagine.")

Verificare overlap între splitturi (image_id):
  Train ∩ Val:  0 imagini
  Train ∩ Test: 0 imagini
  Val ∩ Test:   0 imagini

[OK] ZERO leakage la nivel de imagine.


In [29]:
# Check 3: suma splitturilor = total
total_split = len(train_pool_df) + len(val_df) + len(test_df)
assert total_split == len(df), f"Mismatch: {total_split} != {len(df)}"
print(f"[OK] Toate {len(df):,} imaginile au fost asignate unui split.")

[OK] Toate 25,331 imaginile au fost asignate unui split.


## 8. Verificare distribuție claselor între splitturi

Distribuțiile ar trebui să fie aproape identice (stratify a funcționat).

In [30]:
def class_dist_pct(sub_df):
    return (sub_df['dx'].value_counts(normalize=True) * 100).round(2)

dist = pd.DataFrame({
    'Train (%)':   class_dist_pct(train_pool_df),
    'Val (%)':     class_dist_pct(val_df),
    'Test (%)':    class_dist_pct(test_df),
    'Overall (%)': class_dist_pct(df),
}).fillna(0)

# Diferența max între splitturi (pe clasa)
dist['Max diff'] = (dist[['Train (%)', 'Val (%)', 'Test (%)']].max(axis=1) -
                    dist[['Train (%)', 'Val (%)', 'Test (%)']].min(axis=1)).round(2)

print("Distribuție clase per split (procente):")
dist

Distribuție clase per split (procente):


,Train (%),Val (%),Test (%),Overall (%),Max diff
dx,,,,,
akiec,5.87,5.92,6.13,5.90,0.26
bcc,13.11,13.26,13.05,13.12,0.21
bkl,10.48,9.92,9.79,10.36,0.69
df,0.91,0.95,1.23,0.94,0.32
mel,17.87,18.50,17.03,17.85,1.47
nv,50.73,50.54,51.93,50.83,1.39
vasc,1.03,0.91,0.84,1.00,0.19


In [31]:
# Numere absolute per split × clasă
counts = pd.DataFrame({
    'Train': train_pool_df['dx'].value_counts(),
    'Val':   val_df['dx'].value_counts(),
    'Test':  test_df['dx'].value_counts(),
}).fillna(0).astype(int)

counts['Total'] = counts.sum(axis=1)
counts.loc['TOTAL'] = counts.sum()

print("Numere absolute per split × clasă:")
counts

Numere absolute per split × clasă:


,Train,Val,Test,Total
dx,,,,
akiec,1192,149,154,1495
bcc,2661,334,328,3323
bkl,2128,250,246,2624
df,184,24,31,239
mel,3628,466,428,4522
nv,10297,1273,1305,12875
vasc,209,23,21,253
TOTAL,20299,2519,2513,25331


In [32]:
# Distribuție surse per split (informativ — nu am stratificat pe sursă)
source_dist = pd.DataFrame({
    'Train (%)': (train_pool_df['source'].value_counts(normalize=True) * 100).round(2),
    'Val (%)':   (val_df['source'].value_counts(normalize=True) * 100).round(2),
    'Test (%)':  (test_df['source'].value_counts(normalize=True) * 100).round(2),
}).fillna(0)

print("Distribuție surse per split (informativ):")
source_dist

Distribuție surse per split (informativ):


,Train (%),Val (%),Test (%)
source,,,
BCN_20000,49.11,47.96,49.18
HAM10000,39.62,40.37,38.04
other,8.06,8.46,9.31
MSK,3.21,3.22,3.46


## 9. Salvare CSV-uri

In [33]:
train_pool_path = os.path.join(OUTPUT_DIR, 'train_pool.csv')
val_path        = os.path.join(OUTPUT_DIR, 'val_fixed.csv')
test_path       = os.path.join(OUTPUT_DIR, 'test_fixed.csv')

train_pool_df.to_csv(train_pool_path, index=False)
val_df.to_csv(val_path, index=False)
test_df.to_csv(test_path, index=False)

print(f"[SAVED] {train_pool_path}  ({len(train_pool_df):,} rânduri)")
print(f"[SAVED] {val_path}  ({len(val_df):,} rânduri)")
print(f"[SAVED] {test_path}  ({len(test_df):,} rânduri)")

[SAVED] /content/drive/MyDrive/Thesis/pipeline_outputs/step1/train_pool.csv  (20,299 rânduri)
[SAVED] /content/drive/MyDrive/Thesis/pipeline_outputs/step1/val_fixed.csv  (2,519 rânduri)
[SAVED] /content/drive/MyDrive/Thesis/pipeline_outputs/step1/test_fixed.csv  (2,513 rânduri)


## 10. Generare raport markdown

In [34]:
def generate_report(train_pool_df, val_df, test_df, df, output_path):
    lines = []
    lines.append("# PASUL 1 — Raport split lesion-level")
    lines.append("")
    lines.append(f"**Data generării:** {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}")
    lines.append(f"**Random seed:** {RANDOM_SEED}")
    lines.append("")
    lines.append("## 1. Configurare split")
    lines.append("")
    lines.append(f"- **Strategy:** lesion-level grouping + stratify pe clasă (`dx`)")
    lines.append(f"- **Ratios:** {TRAIN_RATIO} / {VAL_RATIO} / {TEST_RATIO}")
    lines.append(f"- **Total imagini input:** {len(df):,}")
    lines.append(f"- **Total leziuni input:** {df['lesion_id'].nunique():,}")
    lines.append("")
    lines.append("## 2. Rezultate split")
    lines.append("")
    lines.append("| Split | Imagini | Leziuni | % imagini |")
    lines.append("|-------|---------|---------|-----------|")
    for name, sub in [('Train pool', train_pool_df), ('Val', val_df), ('Test', test_df)]:
        n_img = len(sub)
        n_les = sub['lesion_id'].nunique()
        pct = 100 * n_img / len(df)
        lines.append(f"| {name} | {n_img:,} | {n_les:,} | {pct:.2f}% |")
    lines.append(f"| **TOTAL** | **{len(df):,}** | **{df['lesion_id'].nunique():,}** | **100%** |")
    lines.append("")

    lines.append("## 3. Distribuție clase per split")
    lines.append("")
    counts = pd.DataFrame({
        'Train': train_pool_df['dx'].value_counts(),
        'Val':   val_df['dx'].value_counts(),
        'Test':  test_df['dx'].value_counts(),
    }).fillna(0).astype(int)
    counts['Total'] = counts.sum(axis=1)
    lines.append(counts.to_markdown())
    lines.append("")

    lines.append("## 4. Distribuție clase procentual (verificare stratify)")
    lines.append("")
    dist = pd.DataFrame({
        'Train (%)': (train_pool_df['dx'].value_counts(normalize=True) * 100).round(2),
        'Val (%)':   (val_df['dx'].value_counts(normalize=True) * 100).round(2),
        'Test (%)':  (test_df['dx'].value_counts(normalize=True) * 100).round(2),
    }).fillna(0)
    lines.append(dist.to_markdown())
    lines.append("")

    lines.append("## 5. Distribuție surse per split (informativ)")
    lines.append("")
    src_dist = pd.DataFrame({
        'Train (%)': (train_pool_df['source'].value_counts(normalize=True) * 100).round(2),
        'Val (%)':   (val_df['source'].value_counts(normalize=True) * 100).round(2),
        'Test (%)':  (test_df['source'].value_counts(normalize=True) * 100).round(2),
    }).fillna(0)
    lines.append(src_dist.to_markdown())
    lines.append("")
    lines.append("*Notă: Nu s-a făcut stratify pe sursă. Distribuțiile reflectă ce a rezultat natural din stratify-ul pe clasă.*")
    lines.append("")

    lines.append("## 6. Verificări de leakage")
    lines.append("")
    train_lids = set(train_pool_df['lesion_id'])
    val_lids   = set(val_df['lesion_id'])
    test_lids  = set(test_df['lesion_id'])
    lines.append(f"- **Train ∩ Val (lesion_id):** {len(train_lids & val_lids)} (trebuie 0)")
    lines.append(f"- **Train ∩ Test (lesion_id):** {len(train_lids & test_lids)} (trebuie 0)")
    lines.append(f"- **Val ∩ Test (lesion_id):** {len(val_lids & test_lids)} (trebuie 0)")
    lines.append("")

    lines.append("## 7. Decizii și justificare")
    lines.append("")
    lines.append("**De ce 80/10/10 (vs 90/10 al Nawaz et al.)?**")
    lines.append("")
    lines.append("Spre deosebire de splitul 90/10 folosit de Nawaz et al. (care nu separă explicit ")
    lines.append("validation de test, permițând hyperparameter tuning leakage), această lucrare ")
    lines.append("utilizează un split lesion-level stratificat 80/10/10 cu seturi de validation și ")
    lines.append("test distincte. Acest design asigură că performanța finală raportată reflectă ")
    lines.append("generalizare reală, nu adaptare la setul de evaluare.")
    lines.append("")
    lines.append("**De ce stratify pe clasă, nu pe sursă?**")
    lines.append("")
    lines.append("Sursa (HAM/BCN/etc.) e metadata informativă, nu un constraint de evaluare. ")
    lines.append("Stratify-ul pe clasă (`dx`) garantează că performanța per-clasă e comparabilă ")
    lines.append("între splitturi — esențial pentru rapoarte cu metrici macro și balanced accuracy.")
    lines.append("Distribuția surselor în val/test rezultă natural din stratify-ul pe clasă.")
    lines.append("")
    lines.append("**De ce lesion-level grouping?**")
    lines.append("")
    lines.append("HAM10000 conține leziuni cu până la 6 imagini ale aceleași leziuni. Splitul ")
    lines.append("naive la nivel de imagine ar duce la same-lesion leakage (imagini ale aceleași ")
    lines.append("leziuni în atât train cât și test), inflație artificială documentată în ")
    lines.append("Tschandl et al. (2018). Gruparea lesion-level elimină acest risc.")
    lines.append("")

    lines.append("## 8. Output-uri")
    lines.append("")
    lines.append("- `train_pool.csv` — input pentru PASUL 2 (construire tier-uri)")
    lines.append("- `val_fixed.csv` — validation set, FIX pentru toate experimentele")
    lines.append("- `test_fixed.csv` — test set, FIX pentru toate experimentele")
    lines.append("")

    with open(output_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(lines))


report_path = os.path.join(OUTPUT_DIR, 'step1_report.md')
generate_report(train_pool_df, val_df, test_df, df, report_path)
print(f"[SAVED] {report_path}")

[SAVED] /content/drive/MyDrive/Thesis/pipeline_outputs/step1/step1_report.md


## 11. Checklist de validare manuală

Înainte să treci la PASUL 2, verifică în output-uri:

- [ ] Train pool ≈ 80%, Val ≈ 10%, Test ≈ 10% (toleranță ±0.5% datorită lesion grouping)
- [ ] Distribuțiile claselor în Train/Val/Test sunt aproape identice (max diff ~1-2 puncte procentuale)
- [ ] DF (clasa minoritară) e prezent în toate cele 3 splitturi
- [ ] Verificări leakage = 0 (raportate la celulele 7)
- [ ] Total imagini după split = total imagini input

---

## Conexiune cu pașii următori

- **PASUL 2:** Va citi `train_pool.csv` și va construi 5 tier-uri filtrând pe sursă (HAM only → +25% BCN → ... → tot)
- **Val și Test:** rămân fixe peste toate experimentele. Aceleași 2,500+2,500 imagini evaluează toate cele 4 arhitecturi × 5 tier-uri = 20 modele.

## Note pentru thesis writing

În capitolul de metodologie:

> *"The unified dataset (n=24,500 images, ~22,000 unique lesions) was split lesion-level with class-stratified 80/10/10 ratios. All images of a given lesion were assigned to the same split to prevent same-lesion leakage. Stratification was performed on the diagnostic class to ensure comparable per-class performance metrics across splits. The validation and test sets were then frozen for all subsequent experiments, while the training pool was used to construct progressive data tiers (Step 2)."*